# Griffin Library - Intelligent Book Recommendation System

## Notebook 05 · Model - FAISS Index & Recommendation Engine

**Goal:** Build the semantic search index using sentence-transformers and FAISS,
combine it with rating and popularity signals into a hybrid scoring system.

| | Details |
|---|---|
| **Input** | `data/processed/books_features.csv` |
| **Operations** | Text embedding · FAISS index building · Hybrid scoring |
| **Output** | `models/book_index.faiss` · `models/books_cleaned.pkl` |
| **Next Step** | `06_validate.ipynb` - Validate recommendation quality |

---

In [1]:
import sys
# Dependencies installed via requirements.txt

import os
import pickle
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

os.chdir(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
os.makedirs("models", exist_ok=True)

print("Libraries loaded ✓")

C:\Users\[user]\Desktop\Griffin Library\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded ✓


### Load & Validate Features
Load `books_features.csv` and verify all required columns are present.

In [2]:
# ═══════════════════════════════════════════════════════════════
# Load Feature Data
# ═══════════════════════════════════════════════════════════════

df = pd.read_csv("data/processed/books_features.csv")
df = df.reset_index(drop=True)

print(f"Books loaded    : {len(df):,}")
print(f"Columns         : {list(df.columns)}")
print(f"With pages      : {(df['num_pages'] > 0).sum():,} / {len(df):,}")

# Validate required columns
required = ["book_id", "title", "authors", "genres", "avg_rating",
            "weighted_rating", "rating_norm", "num_ratings",
            "popularity_norm", "embedding_text", "tier", "url", "num_pages"]

missing = [c for c in required if c not in df.columns]
if missing:
    print(f"\n⚠️  Missing columns: {missing}")
    print("→ Re-run notebooks 02 and 04 first.")
else:
    print("\n✓ All required columns present")

print(f"\nSample row:")
print(df[["title", "authors", "avg_rating", "num_pages", "tier"]].head(3))

Books loaded    : 8,577
Columns         : ['book_id', 'title', 'authors', 'genres', 'avg_rating', 'weighted_rating', 'rating_norm', 'num_ratings', 'popularity_norm', 'description', 'summary', 'embedding_text', 'tier', 'url', 'num_pages']
With pages      : 1,084 / 8,577

✓ All required columns present

Sample row:
                                               title       authors  \
0                              To Kill a Mockingbird    Harper Lee   
1  Harry Potter and the Philosopher’s Stone (Harr...  J.K. Rowling   
2                                Pride and Prejudice   Jane Austen   

   avg_rating  num_pages  tier  
0        4.27        323     1  
1        4.47          0     2  
2        4.28        333     1  


### Scoring Normalisation
Compute `rating_norm` and `popularity_norm` if not already present.

In [3]:
# ═══════════════════════════════════════════════════════════════
# Ensure Normalised Score Columns Exist
# ═══════════════════════════════════════════════════════════════

if "rating_norm" not in df.columns:
    df["rating_norm"] = (df["avg_rating"] - df["avg_rating"].min()) / \
                        (df["avg_rating"].max() - df["avg_rating"].min())

if "popularity_norm" not in df.columns:
    log_ratings = np.log1p(df["num_ratings"])
    df["popularity_norm"] = (log_ratings - log_ratings.min()) / \
                            (log_ratings.max() - log_ratings.min())

print(f"rating_norm     : min={df['rating_norm'].min():.3f}  max={df['rating_norm'].max():.3f}")
print(f"popularity_norm : min={df['popularity_norm'].min():.3f}  max={df['popularity_norm'].max():.3f}")
print("✓ Normalisation done")

rating_norm     : min=0.000  max=1.000
popularity_norm : min=0.000  max=1.000
✓ Normalisation done


### Build Sentence Embeddings
Encode `embedding_text` using `all-mpnet-base-v2`.
This step takes **5–15 minutes** depending on your machine.

 Model: `all-mpnet-base-v2` - 768-dimensional embeddings, strong semantic understanding.

In [4]:
# ═══════════════════════════════════════════════════════════════
# Build Sentence Embeddings
# ═══════════════════════════════════════════════════════════════

print("Loading model: all-mpnet-base-v2 ...")
model = SentenceTransformer("all-mpnet-base-v2")

texts = df["embedding_text"].fillna("").tolist()

print(f"Encoding {len(texts):,} books — this may take several minutes...")
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,   # L2-normalised → cosine similarity via dot product
    convert_to_numpy=True,
).astype(np.float32)

print(f"\nEmbedding matrix : {embeddings.shape}")
print(f"Dtype            : {embeddings.dtype}")
print("✓ Embeddings built")

Loading model: all-mpnet-base-v2 ...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 4679.94it/s]


Encoding 8,577 books — this may take several minutes...


Batches: 100%|███████████████████████████████████████████████████████████████████████| 135/135 [34:26<00:00, 15.31s/it]



Embedding matrix : (8577, 768)
Dtype            : float32
✓ Embeddings built


### Build FAISS Index
Use `IndexFlatIP` (inner product) — equivalent to cosine similarity on L2-normalised vectors.
Fast exact search, no approximation needed at this dataset size.

In [5]:
# ═══════════════════════════════════════════════════════════════
# Build FAISS Index
# ═══════════════════════════════════════════════════════════════

dim   = embeddings.shape[1]          # 768
index = faiss.IndexFlatIP(dim)        # Inner product = cosine on normalised vecs
index.add(embeddings)

print(f"FAISS index built")
print(f"  Dimension : {dim}")
print(f"  Vectors   : {index.ntotal:,}")
print(f"  Index type: IndexFlatIP (exact search)")

# Quick sanity check
test_vec = model.encode(["a mystery novel"], normalize_embeddings=True).astype(np.float32)
scores, idxs = index.search(test_vec, 3)
print(f"\nSanity check — 'a mystery novel' top 3:")
for score, idx in zip(scores[0], idxs[0]):
    print(f"  {df.iloc[idx]['title'][:50]}  (score: {score:.4f})")
print("\n✓ FAISS index working")

FAISS index built
  Dimension : 768
  Vectors   : 8,577
  Index type: IndexFlatIP (exact search)

Sanity check — 'a mystery novel' top 3:
  Disclaimer  (score: 0.6328)
  Tell Me Everything  (score: 0.5382)
  Goose Island  (score: 0.5341)

✓ FAISS index working


### Save Artifacts
Save the FAISS index and the cleaned DataFrame to `models/`.

In [6]:
# ═══════════════════════════════════════════════════════════════
# Save FAISS Index + DataFrame
# ═══════════════════════════════════════════════════════════════

# Save FAISS index
faiss.write_index(index, "models/book_index.faiss")
print("Saved: models/book_index.faiss")

# Save cleaned DataFrame (only columns needed by the app)
cols_to_keep = [
    "book_id", "title", "authors", "genres",
    "avg_rating", "num_ratings", "num_pages",
    "rating_norm", "popularity_norm",
    "description", "url", "tier",
]
df_clean = df[cols_to_keep].copy()
df_clean.to_pickle("models/books_cleaned.pkl")
print("Saved: models/books_cleaned.pkl")

print(f"\n{'='*50}")
print(f"  Books saved     : {len(df_clean):,}")
print(f"  Columns in pkl  : {list(df_clean.columns)}")
print(f"  With page count : {(df_clean['num_pages'] > 0).sum():,}")
print(f"{'='*50}")
print("\n✓ All artifacts saved — ready for the app!")

Saved: models/book_index.faiss
Saved: models/books_cleaned.pkl

  Books saved     : 8,577
  Columns in pkl  : ['book_id', 'title', 'authors', 'genres', 'avg_rating', 'num_ratings', 'num_pages', 'rating_norm', 'popularity_norm', 'description', 'url', 'tier']
  With page count : 1,084

✓ All artifacts saved — ready for the app!


---
### ✅ Done

| Artifact | Path | Description |
|---|---|---|
| FAISS index | `models/book_index.faiss` | 768-dim semantic vectors |
| Book DataFrame | `models/books_cleaned.pkl` | Metadata + scores + num_pages |

**Next:** Run `06_validate.ipynb` to test recommendation quality.